# SE3082 — Lab 09: Introduction to CUDA

**Student ID:** IT24100315

Based on Lab09_CUDA_Colab_SE3082.pdf (Semester 2, 2026).

**Execution status:** Prepared but not yet executed on a Colab GPU. Expected outputs below are reference values from the lab sheet, not recorded execution results. Run all cells with a T4 GPU and save the resulting notebook before submission.

## Exercise 1 — Set up CUDA in Google Colab

Open this notebook in Colab. Choose **Runtime → Change runtime type → T4 GPU → Save**. Run the diagnostic cells and record the GPU name, compute capability and compiler release from their output. The compile cells use `sm_75` for T4; for L4 use `sm_89`, A100 `sm_80`, or H100 `sm_90` in every compile cell. If `nvcc` is unavailable, use `/usr/local/cuda/bin/nvcc`.

In [ ]:
!nvidia-smi

In [ ]:
!nvcc --version

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap --format=csv

In [ ]:
%%writefile hello.cu
#include <cstdio>

__global__ void hello() {
    printf("Hello from GPU thread %d\n", threadIdx.x);
}

int main(void) {
    hello<<<1, 4>>>();          // 1 block of 4 threads
    cudaDeviceSynchronize();    // wait so the GPU's output is flushed
    return 0;
}


In [ ]:
!nvcc -arch=sm_75 hello.cu -o hello && ./hello

Expected: four GPU greetings for thread IDs 0–3, in any order.

Colab files are temporary; reconnect and run all cells to rebuild them. The notebook itself can be saved to Drive.

## Exercise 2 — Add two integers

One block and one thread add 2 and 7. Expected: `Result is 9`.

In [ ]:
%%writefile ex2_add.cu
#include <cstdio>

// Kernel: this code runs on the device (the NVIDIA GPU)
__global__ void add(int *a, int *b, int *c) {
    *c = *a + *b;
}

// host = CPU, device = GPU
int main(void) {
    int a, b, c;                // host copies of a, b, c
    int *d_a, *d_b, *d_c;       // device copies of a, b, c
    int size = sizeof(int);

    // Allocate space for device copies of a, b, c
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Set up input values
    a = 2;
    b = 7;

    // 1. Copy the inputs from host to device
    cudaMemcpy(d_a, &a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, &b, size, cudaMemcpyHostToDevice);

    // 2. Launch the add() kernel on the GPU: 1 block, 1 thread
    add<<<1, 1>>>(d_a, d_b, d_c);

    // Colab edition: report a failed launch instead of silently printing garbage
    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    // 3. Copy the result from device to host
    cudaMemcpy(&c, d_c, size, cudaMemcpyDeviceToHost);
    printf("Result is %d\n", c);

    // Cleanup
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}


In [ ]:
!nvcc -arch=sm_75 ex2_add.cu -o ex2_add && ./ex2_add

### Think — answers

1. In this program, ordinary host variables reside in CPU memory. `cudaMalloc` allocates separate GPU storage accessible to the kernel; explicit copies transfer the values.
2. `&a` is the address of the host integer to copy; `d_a` already contains the address of the device allocation.
3. `add<<<1, 1>>>` runs one thread. `add<<<1, 10>>>` runs ten threads, but they would all write to the same `c`, creating a data race even though they compute the same value. Use separate output elements or guard the scalar write to thread 0.

## Exercise 3 — 512 blocks, one thread per block

Predict: 512 blocks × 1 thread = 512 threads. Each block needs a warp with only one active lane, leaving 31 lanes unused. Expected final line: `Verification: 0 mismatches out of 512`.

In [ ]:
%%writefile ex3_blocks.cu
#include <cstdio>
#include <cstdlib>

#define N 512

__global__ void add(int *a, int *b, int *c) {
    // each block runs only one addition
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;             // host copies of a, b, c
    int *d_a, *d_b, *d_c;       // device copies of a, b, c
    int size = N * sizeof(int);

    // Allocate space for device copies of a, b, c
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate space for host copies of a, b, c and set up input values
    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    // Copy inputs to device
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // Launch add() kernel on GPU with N blocks, 1 thread per block
    add<<<N, 1>>>(d_a, d_b, d_c);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    // Copy result back to host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int r = 0; r < N; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);

    // Check the GPU's answers against the CPU
    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);

    // Cleanup
    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}


In [ ]:
!nvcc -arch=sm_75 ex3_blocks.cu -o ex3_blocks && ./ex3_blocks

## Exercise 4 — One block, 512 threads

`threadIdx.x` selects each element. Expected: `0) 83 + 77 = 160`, last result `511) 67 + 6 = 73`, and zero mismatches. A single block has 16 full warps, but cannot distribute work across multiple multiprocessors.

In [ ]:
%%writefile ex4_threads.cu
#include <cstdio>
#include <cstdlib>

#define N 512

__global__ void addT(int *a, int *b, int *c) {
    c[threadIdx.x] = a[threadIdx.x] + b[threadIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;             // host copies of a, b, c
    int *d_a, *d_b, *d_c;       // device copies of a, b, c
    int size = N * sizeof(int);

    // Allocate space for device copies of a, b, c
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate space for host copies of a, b, c and set up input values
    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    // Copy inputs to device
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // Launch addT() kernel on GPU with 1 block of N threads
    addT<<<1, N>>>(d_a, d_b, d_c);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    // Copy result back to host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int r = 0; r < N; r++)
        printf("%d) %d + %d = %d\n", r, a[r], b[r], c[r]);

    // Check the GPU's answers against the CPU
    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);

    // Cleanup
    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}


In [ ]:
!nvcc -arch=sm_75 ex4_threads.cu -o ex4_threads && ./ex4_threads

### Experiment — 2,048 threads

The following cells make a temporary copy with N = 2048, compile and run it, then remove it. The original Exercise 4 remains at N = 512. Expected launch error: `Kernel launch failed: invalid configuration argument`, reported by `cudaGetLastError()`, because 2048 exceeds the 1024 threads-per-block limit. Result lines after a failed launch are invalid because no results were computed.

In [ ]:
from pathlib import Path
p = Path('ex4_threads.cu')
Path('ex4_limit.cu').write_text(p.read_text().replace('#define N 512', '#define N 2048'))

In [ ]:
!nvcc -arch=sm_75 ex4_limit.cu -o ex4_limit && ./ex4_limit

In [ ]:
from pathlib import Path
for name in ('ex4_limit.cu', 'ex4_limit'):
    Path(name).unlink(missing_ok=True)

## Exercise 5 — Multiply 10,000,000 vector elements

Completed global index, bounds guard, rounded-up block count, kernel launch, copy back, last 1,000 results and full CPU verification.

Expected: 19,532 blocks × 512 = 10,000,384 threads; the final block has 128 active threads and 384 guarded threads. The printed results run from `9999000) 76 x 52 = 3952` to `9999999) 31 x 73 = 2263`, followed by `Verification: 0 mismatches out of 10000000`.

In [ ]:
%%writefile ex5_vecmul.cu
#include <cstdio>
#include <cstdlib>

#define N 10000000              // 10 million elements
#define THREADS_PER_BLOCK 512

__global__ void vecMul(int *a, int *b, int *c, int n) {
    // Solution: global index of this thread (use blockIdx, blockDim, threadIdx)
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    // Solution: only compute if i is inside the array. Why is this needed?
    if (i < n)
        c[i] = a[i] * b[i];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;             // host copies of a, b, c
    int *d_a, *d_b, *d_c;       // device copies of a, b, c
    size_t size = (size_t)N * sizeof(int);

    // Allocate device memory
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate host memory and generate the input values
    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    // Copy inputs to device
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // Solution: number of blocks needed to cover N elements (round UP)
    int blocks = (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;
    printf("Launching %d blocks x %d threads\n", blocks, THREADS_PER_BLOCK);

    vecMul<<<blocks, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, N);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int i = N - 1000; i < N; i++)
        printf("%d) %d x %d = %d\n", i, a[i], b[i], c[i]);

    int errors = 0;
    for (int i = 0; i < N; i++)
        if (c[i] != a[i] * b[i]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);

    // Cleanup
    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}


In [ ]:
!nvcc -arch=sm_75 ex5_vecmul.cu -o ex5_vecmul && ./ex5_vecmul

### Think — answers

A multiple of 32 fills complete warps. With 500 threads per block, 20,000 blocks launch exactly 10,000,000 threads, but each block uses 16 warps with 12 unused lanes in its last warp. A bounds guard is still appropriate for arbitrary array sizes.

## Exercise 6 — Element-wise 10,000 × 10,000 matrix product

Columns use x; rows use y. Flat index is `row * cols + col`. Both coordinates are guarded. Blocks have 32 × 16 = 512 threads; the rounded-up grid is 313 × 625 = 195,625 blocks and 100,160,000 threads. The extra 160,000 threads are guarded. Each matrix occupies 400 MB; three matrices require 1.2 GB on both host and device.

Expected first printed element: `C[9999][9000]) 29 x 56 = 1624`; last: `C[9999][9999]) 86 x 21 = 1806`; verification: zero mismatches out of 100,000,000.

In [ ]:
%%writefile ex6_matrix.cu
#include <cstdio>
#include <cstdlib>

#define ROWS 10000
#define COLS 10000

// Element-wise product: C[r][c] = A[r][c] * B[r][c]  (NOT matrix multiplication)
__global__ void mulElem(int *a, int *b, int *c, int rows, int cols) {
    // Solution: row comes from the y dimension, col from the x dimension
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    // Solution: only compute if BOTH row and col are inside the matrix
    // Solution: convert (row, col) into a flat index: row * cols + col
    if (row < rows && col < cols) {
        int idx = row * cols + col;
        c[idx] = a[idx] * b[idx];
    }
}

void random_ints(int *x, size_t size) {
    for (size_t i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;             // host copies (flat arrays, row-major)
    int *d_a, *d_b, *d_c;       // device copies
    size_t count = (size_t)ROWS * COLS;
    size_t size  = count * sizeof(int);         // 400 MB per matrix

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size); random_ints(a, count);
    b = (int *)malloc(size); random_ints(b, count);
    c = (int *)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // 32 x 16 = 512 threads per block
    dim3 threadsPerBlock(32, 16);
    // Solution: blocks needed in x (columns) and in y (rows), rounding UP
    dim3 numBlocks((COLS + threadsPerBlock.x - 1) / threadsPerBlock.x,
                   (ROWS + threadsPerBlock.y - 1) / threadsPerBlock.y);
    printf("Grid: %d x %d blocks\n", numBlocks.x, numBlocks.y);

    mulElem<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, ROWS, COLS);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int col = COLS - 1000; col < COLS; col++) {
        size_t idx = (size_t)(ROWS - 1) * COLS + col;
        printf("C[%d][%d]) %d x %d = %d\n", ROWS - 1, col, a[idx], b[idx], c[idx]);
    }

    size_t errors = 0;
    for (size_t i = 0; i < count; i++)
        if (c[i] != a[i] * b[i]) errors++;
    printf("Verification: %zu mismatches out of %zu\n", errors, count);

    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}


In [ ]:
!nvcc -arch=sm_75 ex6_matrix.cu -o ex6_matrix && ./ex6_matrix

## Finishing checklist

- Run all cells on a Colab T4 GPU and inspect the actual outputs.
- Confirm hello prints four greetings, Exercise 2 gives 9, and Exercises 3–6 report zero mismatches.
- Confirm the Exercise 4 limit experiment reports the expected launch failure.
- Save the notebook with outputs visible; download the `.ipynb` and submit or share as instructed.
- Choose **Runtime → Disconnect and delete runtime** to release the GPU.